<a href="https://colab.research.google.com/github/DanielBCunha/Fundamentos-de-Docking-Molecular---SBBS/blob/main/Fundamentos_de_Docking_Molecular_SBBS_Aula_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1: Instalação de dependências

#@markdown > Instala a livraria pandas para python

!pip install pandas
!pip install pdb2pqr
!pip install meeko gemmi prody
!pip install rdkit

import sys, platform
from pathlib import Path

def locate_file(from_path = None, query_path = None, query_name = "query file"):

    possible_path = list(from_path.glob(query_path))

    if not possible_path:
        raise FileNotFoundError(f"Cannot find {query_name} from {from_path} by {query_path}")

    return_which = (
        f"using {query_name} at:\n"
        f"{possible_path[0]}\n"
    )
    print(return_which)

    return possible_path[0]


mk_prepare_ligand = locate_file(from_path = Path("/usr/local/bin"), query_path = "mk_prepare_ligand.py", query_name = "mk_prepare_ligand.py")
mk_prepare_receptor = locate_file(from_path = Path("/usr/local/bin"), query_path = "mk_prepare_receptor.py", query_name = "mk_prepare_receptor.py")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pdb2pqr]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 56.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.9 MB/s  0:00:00
  Created wheel for prody: filename=prody-2.6.1-cp313-cp313-linux_x86_64.whl size=36502964 sha256=cf1bf1c6cf1949ddfe36f40a48f777da747ea5f591cf278a40e804badc30da26
  Stored in directory: /root/.cache/pip/wheels/88/35/fd/abca580f4f308fee04dc989ca1ad6b0d9bfc26b19f83a7c682
Successfully built prody
  Attempting uninstall: pyparsing
    Found existing installation: pyparsing 3.3.3
    Uninstalling pyparsing-3.3.3:
      Successfully uninstalled pyparsing-3.3.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 


--------------------------------------------------------


In [2]:
# @title 2: Download e preparo da estrutura do receptor

#@markdown > Insira código PDB abaixo

pdb_token = "4JTU" #@param {type:"string"}
! curl "http://files.rcsb.org/view/{pdb_token}.pdb" -o "{pdb_token}.pdb"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  291k    0  291k    0     0   444k      0 --:--:-- --:--:-- --:--:--  444k


In [2]:
!cat 4JTU.pdb

HEADER    OXIDOREDUCTASE                          24-MAR-13   4JTU              
TITLE     CRYSTAL STRUCTURE OF HUMAN DIHYDROOROTATE DEHYDROGENASE (DHODH) WITH  
TITLE    2 BREQUINAR ANALOGUE                                                   
COMPND    MOL_ID: 1;                                                            
COMPND   2 MOLECULE: DIHYDROOROTATE DEHYDROGENASE (QUINONE), MITOCHONDRIAL;     
COMPND   3 CHAIN: A;                                                            
COMPND   4 FRAGMENT: UNP RESIDUES 29-395;                                       
COMPND   5 SYNONYM: DHODEHASE, DIHYDROOROTATE OXIDASE;                          
COMPND   6 EC: 1.3.5.2;                                                         
COMPND   7 ENGINEERED: YES                                                      
SOURCE    MOL_ID: 1;                                                            
SOURCE   2 ORGANISM_SCIENTIFIC: HOMO SAPIENS;                                   
SOURCE   3 ORGANISM_COMMON: 

In [3]:
!grep "^ATOM  " 4JTU.pdb > 4JTU_clean.pdb

In [ ]:
!cat 4JTU_clean.pdb

In [4]:
!pdb2pqr --ff AMBER --titration-state-method propka --with-ph 7.4 4JTU_clean.pdb 4JTU_prepared_pH74.pdb

INFO:PDB2PQR v3.8.0: biomolecular structure conversion software.
INFO:Please cite:  Jurrus E, et al.  Improvements to the APBS biomolecular solvation software suite.  Protein Sci 27 112-128 (2018).
INFO:Please cite:  Dolinsky TJ, et al.  PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. Nucleic Acids Res 35 W522-W525 (2007).
INFO:Checking and transforming input arguments.
INFO:Loading topology files.
INFO:Loading molecule: 4JTU_clean.pdb
INFO:Setting up molecule.
INFO:Created biomolecule object with 365 residues and 2792 atoms.
INFO:Setting termini states for biomolecule chains.
INFO:Loading forcefield.
INFO:Loading hydrogen topology definitions.
INFO:This biomolecule is clean.  No repair needed.
INFO:Updating disulfide bridges.
INFO:Debumping biomolecule.
INFO:Assigning titration states with PROPKA.
INFO:
propka3.5.1                                                                                  2026-11-11

------------------

In [5]:
!grep -E '^HETATM' 4JTU.pdb | grep ' FMN ' > fmn.pdb

In [6]:
# @title

#@markdown > Adiciona hidrogênios ao FMN utilizando o contexto químico do SMILES

from rdkit import Chem
from rdkit.Chem import AllChem

tmpl = Chem.MolFromSmiles(
    "CC1=CC2=C(C=C1C)N(C3=NC(=O)NC(=O)C3=N2)CC(C(C(COP(=O)([O-])[O-])O)O)O")
pdb = Chem.MolFromPDBFile("fmn.pdb", removeHs=True, sanitize=False, proximityBonding=True)
mol = AllChem.AssignBondOrdersFromTemplate(tmpl, pdb)
mh = Chem.AddHs(mol, addCoords=True, addResidueInfo=True)
Chem.MolToPDBFile(mh, "fmn_H.pdb")

[12:14:45] WARNING: More than one matching pattern found - picking one



In [7]:
!grep -E '^(ATOM|TER)' 4JTU_prepared_pH74.pdb > prepared_receptor.pdb
!grep -E '^HETATM' fmn_H.pdb >> prepared_receptor.pdb
!echo END >> prepared_receptor.pdb

In [50]:
!python {mk_prepare_receptor} --help

usage: mk_prepare_receptor.py [-h] [--read_pdb PDB_FILENAME]
                              [--read_json JSON_FILENAME]
                              [--read_pqr PQR_FILENAME] [-i MACROMOL_FILENAME]
                              [-o OUTPUT_BASENAME] [-p [PDBQT_FILENAME ...]]
                              [-j [JSON_FILENAME ...]]
                              [--write_pdb [PDB_FILENAME ...]]
                              [--ignore_https_cert] [-g [GPF_FILENAME ...]]
                              [-v [VINA_BOX_FILENAME ...]]
                              [--debug_fn DEBUG_FN] [-n SET_TEMPLATE]
                              [-d DELETE_RESIDUES] [-b BLUNT_ENDS]
                              [--add_templates ADD_TEMPLATES]
                              [--cache_templates [CACHE_TEMPLATES]]
                              [--config_file CONFIG_FILE]
                              [--config_preset CONFIG_PRESET] [-x]
                              [--delete_bad_res_from_box_radius DELETE_BAD_RES_F

In [ ]:
args = ""
import prody as pr

!python {mk_prepare_receptor} -i prepared_receptor.pdb -o receptor -p

In [8]:
args = ""
import prody as pr
def sanitize_pdb_elements(input_pdb, output_pdb):
    cleaned_lines = []
    with open(input_pdb, 'r') as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                # Extract atom name to deduce the correct element symbol
                atom_name = line[12:16].strip()
                if atom_name.startswith("H") or (len(atom_name) > 1 and atom_name[1] == "H" and atom_name[0].isdigit()):
                    element = "H"
                elif atom_name.startswith("C"):
                    element = "C"
                elif atom_name.startswith("N"):
                    element = "N"
                elif atom_name.startswith("O"):
                    element = "O"
                elif atom_name.startswith("P"):
                    element = "P"
                elif atom_name.startswith("S"):
                    element = "S"
                else:
                    element = atom_name[0] # Fallback

                # Ensure the line has 80 characters, and place the element symbol at columns 77-78
                line_padded = f"{line.rstrip('\n'):<80}"
                line = line_padded[:76] + f"{element:>2}" + line_padded[78:]
            cleaned_lines.append(line.rstrip('\n') + '\n')

    with open(output_pdb, 'w') as f:
        f.writelines(cleaned_lines)
    print(f"Successfully sanitized {input_pdb} and saved as {output_pdb}")

pr.confProDy(verbosity='none')
sanitize_pdb_elements("prepared_receptor.pdb", "prepared_receptor_clean.pdb")

!python {mk_prepare_receptor} -i prepared_receptor_clean.pdb -o receptor -p

@> ProDy is configured: verbosity='none'
INFO:.prody:ProDy is configured: verbosity='none'


Successfully sanitized prepared_receptor.pdb and saved as prepared_receptor_clean.pdb
Template padding will be used.
 gasteiger charges will be read from template file 
 
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1237: RuntimeWarning: Input residues {'A:401': 'FMN'} not in residue_templates

  warnings.warn(err, RuntimeWarning)
/usr/local/lib/python3.13/dist-packages/meeko/polymer.py:1238: RuntimeWarning: Trying to resolve unknown residues by building chemical templates... 
  warnings.warn("Trying to resolve unknown residues by building chemical templates... ", RuntimeWarning)

Files written:
receptor.pdbqt <-- static (i.e., rigid) receptor input file



---------------------------------------------------------


In [9]:
# @title 3: Preparação de livrarias de ligantes

#@markdown > Livraria de ligantes e script de extração

!wget https://raw.githubusercontent.com/DanielBCunha/Fundamentos-de-Docking-Molecular---SBBS/refs/heads/main/extract_ligands.py
!wget https://raw.githubusercontent.com/DanielBCunha/Fundamentos-de-Docking-Molecular---SBBS/refs/heads/main/ligands.tsv

--2026-11-11 12:03:10--  https://raw.githubusercontent.com/DanielBCunha/Fundamentos-de-Docking-Molecular---SBBS/refs/heads/main/extract_ligands.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 988 [text/plain]
Saving to: ‘extract_ligands.py’

extract_ligands.py  100%[===================>]     988  --.-KB/s    in 0s      

2026-11-11 12:03:10 (33.4 MB/s) - ‘extract_ligands.py’ saved [988/988]

--2026-11-11 12:03:10--  https://raw.githubusercontent.com/DanielBCunha/Fundamentos-de-Docking-Molecular---SBBS/refs/heads/main/ligands.tsv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP 

In [20]:
!cat ligands.tsv

"Molecule ChEMBL ID"	"Molecule Name"	"Molecule Max Phase"	"Molecular Weight"	"#RO5 Violations"	"AlogP"	"Compound Key"	"Smiles"	"Standard Type"	"Standard Relation"	"Standard Value"	"Standard Units"	"pChEMBL Value"	"Data Validity Comment"	"Comment"	"Uo Units"	"Ligand Efficiency BEI"	"Ligand Efficiency LE"	"Ligand Efficiency LLE"	"Ligand Efficiency SEI"	"Potential Duplicate"	"Assay ChEMBL ID"	"Assay Description"	"Assay Type"	"BAO Format ID"	"BAO Label"	"Assay Organism"	"Assay Tissue ChEMBL ID"	"Assay Tissue Name"	"Assay Cell Type"	"Assay Subcellular Fraction"	"Assay Parameters"	"Assay Variant Accession"	"Assay Variant Mutation"	"Target ChEMBL ID"	"Target Name"	"Target Organism"	"Target Type"	"Document ChEMBL ID"	"Source ID"	"Source Name"	"Document Journal"	"Document Year"	"Cell ChEMBL ID"	"Properties"	"Action Type"	"Standard Text Value"	"Value"	"Modality"
"CHEMBL973"	"TERIFLUNOMIDE"	"4.0"	"270.21"	"0"	"3.00"	"3, (A-771726)"	"C/C(O)=C(\C#N)C(=O)Nc1ccc(C(F)(F)F)cc1"	"Ki"	"'='"	"32.0"	"nM"	"

In [10]:
!cat extract_ligands.py

import pandas as pd
from pathlib import Path

# Input file
INPUT_FILE = "ligands.tsv"

# Output directories
OUTPUT_DIR = Path("ligands")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read TSV file
df = pd.read_csv(INPUT_FILE, sep="\t")

# Convert Standard Value to numeric
df["Standard Value"] = pd.to_numeric(
    df["Standard Value"], errors="coerce"
)

# Separate active and inactive compounds
active = df[df["Standard Value"] < 50]
inactive = df[df["Standard Value"] > 50]

# Export in SMILES ChEMBL_ID format
active[["Smiles", "Molecule ChEMBL ID"]].to_csv(
    OUTPUT_DIR / "actives.smi",
    sep=" ",
    index=False,
    header=False
)

inactive[["Smiles", "Molecule ChEMBL ID"]].to_csv(
    OUTPUT_DIR / "inactives.smi",
    sep=" ",
    index=False,
    header=False
)

# Print summary
print(f"Total compounds: {len(df)}")
print(f"Active compounds (<50): {len(active)}")
print(f"Inactive compounds (>50): {len(inactive)}")
print(f"\nOutput saved to: {OUTPUT_DIR.resolve()}")
